In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_107_Pusa_Delhi_IMD_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,137.03,221.00,7.74,24.54,32.28,NaN,NaN,1.89,30.12,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,139.65,217.45,8.64,25.92,34.56,NaN,NaN,1.68,30.21,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,142.59,224.14,12.73,31.09,43.82,NaN,NaN,2.07,29.19,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2024-01-04,173.48,260.36,11.76,26.68,38.44,NaN,NaN,2.12,27.41,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2024-01-05,150.52,247.65,13.05,28.37,41.41,NaN,NaN,1.70,26.97,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,100.17,168.03,22.89,49.69,45.03,NaN,NaN,1.95,14.41,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
362,2024-12-28,60.56,101.40,19.08,33.17,33.16,NaN,NaN,1.74,12.90,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
363,2024-12-29,63.68,109.14,4.63,14.13,11.28,NaN,NaN,1.52,22.05,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
364,2024-12-30,89.96,112.41,3.61,14.19,10.48,NaN,NaN,1.50,29.72,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 11)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['NH3 (µg/m³)', 'Toluene (µg/m³)']
Dropped rows (>70% NaN): 1
Missing values after imputation:
 Timestamp        0
PM2.5 (µg/m³)    0
PM10 (µg/m³)     0
NO (µg/m³)       0
NO2 (µg/m³)      0
NOx (ppb)        0
CO (mg/m³)       0
Ozone (µg/m³)    0
TOT-RF (mm)      0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (365, 9)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         137.03        221.00        7.74        24.54   
1  2024-01-02         139.65        217.45        8.64        25.92   
2  2024-01-03         142.59        224.14       12.73        31.09   
3  2024-01-04         173.48        260.36       11.76        26.68   
4  2024-01-05         150.52        247.65       13.05        28.37   

   NOx (ppb)  CO (mg/m³)  Ozone (µg/m³)  TOT-RF (mm)  
0      32.28        1.89          30.12          0.0  
1      34.56        1.68          30.21          0.0  
2      43.82        2.07          29.19          0.0  
3      38.44        2.12          27.41          0.0  
4      41.41        1.70          26.97          0.0  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),CO (mg/m³),Ozone (µg/m³),TOT-RF (mm)
0,2024-01-01,1.007765,0.587383,-0.825327,0.137581,-0.270943,0.805385,1.111905,0.0
1,2024-01-02,1.060684,0.546366,-0.741758,0.229202,-0.175280,0.379495,1.123382,0.0
2,2024-01-03,1.120067,0.623663,-0.361984,0.572448,0.213244,1.170434,0.993314,0.0
3,2024-01-04,1.743989,1.042151,-0.452053,0.279660,-0.012486,1.271836,0.766332,0.0
4,2024-01-05,1.280238,0.895298,-0.332271,0.391862,0.112127,0.420056,0.710225,0.0
...,...,...,...,...,...,...,...,...,...
360,2024-12-27,0.263260,-0.024635,0.581415,1.807340,0.264012,0.927068,-0.891398,0.0
361,2024-12-28,-0.536790,-0.794482,0.227641,0.710544,-0.234020,0.501178,-1.083949,0.0
362,2024-12-29,-0.473772,-0.705054,-1.114104,-0.553560,-1.152044,0.055007,0.082838,0.0
363,2024-12-30,0.057036,-0.667272,-1.208815,-0.549577,-1.185610,0.014446,1.060898,0.0


In [10]:
df.to_excel('PusaIMD2024.xlsx', index=False)